# Step 6 — NammaMart ELT pipeline test

This notebook runs the ELT implementation locally. The raw layer is loaded from the CSV files as text; staging and mart transformations are executed from the checked-in SQL files.

Flow: load raw → transform staging with SQL → build marts with SQL → inspect row counts and business outputs.

In [1]:
from pathlib import Path
import sys
import duckdb

# Import the same ELT module used by the Airflow DAG.
sys.path.insert(0, str(Path('include').resolve()))
from nammamart_elt import run_elt

# Keep the ELT warehouse separate from the ETL warehouse for comparison.
warehouse = Path('assignment/warehouse_samy_elt_step6.duckdb')
metrics = run_elt(
    warehouse_path=warehouse,
    run_id='step6-local-test',
)
for name, value in metrics.items():
    print(f'{name}: {value:,}' if isinstance(value, int) else f'{name}: {value}')

warehouse_path: assignment/warehouse_samy_elt_step6.duckdb
raw_orders: 2,412
stg_orders: 2,412
passing_orders: 2,304
mart_store_rows: 386
mart_category_rows: 6
reconciliation_rows: 204


In [2]:
# Inspect every layer to prove that raw data, staging decisions and marts exist.
con = duckdb.connect(str(warehouse), read_only=True)
display(con.execute("""
    SELECT table_name, estimated_size
    FROM duckdb_tables()
    WHERE schema_name = 'main'
    ORDER BY table_name
""").fetchdf())

# Raw rows are retained while staging exposes pass/fail decisions.
display(con.execute("SELECT dq_status, count(*) AS rows FROM stg_orders GROUP BY dq_status").fetchdf())
display(con.execute("SELECT * FROM mart_category_margin ORDER BY category").fetchdf())
display(con.execute("SELECT * FROM mart_daily_store_kpis ORDER BY order_date, zone, store_id LIMIT 10").fetchdf())
display(con.execute("SELECT * FROM mart_top_customers LIMIT 10").fetchdf())
con.close()

,table_name,estimated_size
0,mart_category_margin,6
1,mart_daily_store_kpis,386
2,mart_delivery_sla,15
3,mart_payment_reconciliation,204
4,mart_top_customers,10
5,raw_customers,405
6,raw_orders,2412
7,raw_payments,2228
8,raw_products,152
9,raw_stores,15


,dq_status,rows
0,PASS,2304
1,FAIL,108


,category,revenue,gross_margin
0,Dairy & Bakery,168320.35,44330.09
1,Fruits & Vegetables,87627.40,21064.58
2,Household,198026.35,51065.44
3,Personal Care,201484.20,44555.02
4,Snacks & Beverages,161660.85,37652.51
5,Staples,251129.35,62914.16


,order_date,zone,store_id,daily_net_revenue,delivered_orders
0,2026-09-01,East,S08,2984.70,4
1,2026-09-01,East,S09,2945.40,6
2,2026-09-01,East,S10,2141.60,3
3,2026-09-01,East,S11,2121.15,6
4,2026-09-01,North,S01,2247.80,4
5,2026-09-01,North,S02,2385.55,6
6,2026-09-01,North,S03,4583.80,6
7,2026-09-01,South,S04,1271.30,2
8,2026-09-01,South,S05,803.00,2
9,2026-09-01,South,S06,2812.00,6


,customer_id,full_name,email_masked,phone_masked,loyalty_tier,total_spend
0,C00136,Sanjay Bhat,***,96******88,BRONZE,8738.10
1,C00236,Sneha Naidu,***,90******01,PLATINUM,8132.40
2,C00283,Nikhil Rao,***,91******43,BRONZE,7934.60
3,C00071,Mahesh Pillai,***,91******42,SILVER,7349.30
4,C00310,Kiran Patil,***,91******35,BRONZE,7258.95
5,C00209,Kavya Pillai,***,98******97,PLATINUM,6519.40
6,C00338,Bhavana Iyer,***,94******05,SILVER,6506.55
7,C00115,Sanjay Kumar,***,96******85,BRONZE,6483.70
8,C00036,Deepa Desai,***,90******55,BRONZE,6466.50
9,C00072,Nandini Murthy,***,98******05,SILVER,6433.95


In [3]:
# Re-running the initial ELT load replaces the derived tables, so this test
# confirms the initial pipeline does not duplicate rows on rerun.
second_metrics = run_elt(warehouse_path=warehouse, run_id='step6-local-rerun')
con = duckdb.connect(str(warehouse), read_only=True)
raw_count = con.execute('SELECT count(*) FROM raw_orders').fetchone()[0]
stg_count = con.execute('SELECT count(*) FROM stg_orders').fetchone()[0]
con.close()
print(f'Raw orders after rerun: {raw_count:,}')
print(f'Staging orders after rerun: {stg_count:,}')
print('The initial ELT run is deterministic; incremental append/merge is implemented in the next step.')

Raw orders after rerun: 2,412
Staging orders after rerun: 2,412
The initial ELT run is deterministic; incremental append/merge is implemented in the next step.
